In [25]:
import pandas as pd
import numpy as np
import os

In [26]:
src_file = os.path.join('..\\data\\raw\\btcusdt-4h-2023-2026.csv')
df = pd.read_csv(src_file)
df.head()

,Timestamp,Open,High,Low,Close,Volume
0,2023-01-01 00:00:00+00:00,16541.77,16559.77,16508.39,16533.04,15515.82327
1,2023-01-01 04:00:00+00:00,16533.04,16550.00,16499.01,16526.19,16532.24115
2,2023-01-01 08:00:00+00:00,16525.70,16557.00,16505.20,16556.66,15915.96701
3,2023-01-01 12:00:00+00:00,16556.66,16572.94,16533.68,16558.73,15046.09096
4,2023-01-01 16:00:00+00:00,16558.73,16623.65,16558.00,16603.08,18532.64857


In [27]:
df['RollingMax'] = df['Close'].rolling(window=14).max()
df['RollingMax_Shifted'] = df['RollingMax'].shift(1)
df['Breakout_Bullish'] = df['Close'] > df['RollingMax_Shifted'] 
df['AvgVolume'] = df['Volume'].rolling(window=14).mean()
df['AvgVolume_Shifted'] = df['AvgVolume'].shift(1)
df['Volume_Confirmed'] = df['Volume'] > df['AvgVolume_Shifted'] * 1.5
df['Real_Breakout'] = df['Volume_Confirmed'] & df['Breakout_Bullish']
df

,Timestamp,Open,High,Low,Close,Volume,RollingMax,RollingMax_Shifted,Breakout_Bullish,AvgVolume,AvgVolume_Shifted,Volume_Confirmed,Real_Breakout
0,2023-01-01 00:00:00+00:00,16541.77,16559.77,16508.39,16533.04,15515.82327,NaN,NaN,False,NaN,NaN,False,False
1,2023-01-01 04:00:00+00:00,16533.04,16550.00,16499.01,16526.19,16532.24115,NaN,NaN,False,NaN,NaN,False,False
2,2023-01-01 08:00:00+00:00,16525.70,16557.00,16505.20,16556.66,15915.96701,NaN,NaN,False,NaN,NaN,False,False
3,2023-01-01 12:00:00+00:00,16556.66,16572.94,16533.68,16558.73,15046.09096,NaN,NaN,False,NaN,NaN,False,False
4,2023-01-01 16:00:00+00:00,16558.73,16623.65,16558.00,16603.08,18532.64857,NaN,NaN,False,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7755,2026-07-16 12:00:00+00:00,64256.52,64896.00,63838.28,64704.73,3169.90689,65427.61,65427.61,False,3598.025655,3520.770625,False,False
7756,2026-07-16 16:00:00+00:00,64704.73,64712.00,63984.09,64271.84,1783.88455,65427.61,65427.61,False,3601.693357,3598.025655,False,False
7757,2026-07-16 20:00:00+00:00,64271.85,64276.00,63748.74,63830.20,1224.95560,65427.61,65427.61,False,3060.860107,3601.693357,False,False
7758,2026-07-17 00:00:00+00:00,63830.20,64067.69,63380.28,63570.00,2665.58621,65427.61,65427.61,False,3016.237659,3060.860107,False,False


In [28]:
df['Prev_Close'] = df['Close'].shift(1)
df['TR1'] = df['High'] - df['Low']
df['TR2'] = (df['High'] - df['Prev_Close']).abs()
df['TR3'] = (df['Low'] - df['Prev_Close']).abs()
df['TrueRange'] = df[['TR1', 'TR2', 'TR3']].max(axis=1)
df['ATR'] = df['TrueRange'].rolling(window=14).mean()
df['ATR_Shifted'] = df['ATR'].shift(1)
df.head(30)

,Timestamp,Open,High,Low,Close,Volume,RollingMax,RollingMax_Shifted,Breakout_Bullish,AvgVolume,AvgVolume_Shifted,Volume_Confirmed,Real_Breakout,Prev_Close,TR1,TR2,TR3,TrueRange,ATR,ATR_Shifted
0,2023-01-01 00:00:00+00:00,16541.77,16559.77,16508.39,16533.04,15515.82327,NaN,NaN,False,NaN,NaN,False,False,NaN,51.38,NaN,NaN,51.38,NaN,NaN
1,2023-01-01 04:00:00+00:00,16533.04,16550.00,16499.01,16526.19,16532.24115,NaN,NaN,False,NaN,NaN,False,False,16533.04,50.99,16.96,34.03,50.99,NaN,NaN
2,2023-01-01 08:00:00+00:00,16525.70,16557.00,16505.20,16556.66,15915.96701,NaN,NaN,False,NaN,NaN,False,False,16526.19,51.80,30.81,20.99,51.80,NaN,NaN
3,2023-01-01 12:00:00+00:00,16556.66,16572.94,16533.68,16558.73,15046.09096,NaN,NaN,False,NaN,NaN,False,False,16556.66,39.26,16.28,22.98,39.26,NaN,NaN
4,2023-01-01 16:00:00+00:00,16558.73,16623.65,16558.00,16603.08,18532.64857,NaN,NaN,False,NaN,NaN,False,False,16558.73,65.65,64.92,0.73,65.65,NaN,NaN
5,2023-01-01 20:00:00+00:00,16603.53,16628.00,16586.24,16616.75,15382.64278,NaN,NaN,False,NaN,NaN,False,False,16603.08,41.76,24.92,16.84,41.76,NaN,NaN
6,2023-01-02 00:00:00+00:00,16617.17,16707.25,16548.70,16661.94,19463.47510,NaN,NaN,False,NaN,NaN,False,False,16616.75,158.55,90.50,68.05,158.55,NaN,NaN
7,2023-01-02 04:00:00+00:00,16662.38,16769.51,16619.44,16721.28,26026.88178,NaN,NaN,False,NaN,NaN,False,False,16661.94,150.07,107.57,42.50,150.07,NaN,NaN
8,2023-01-02 08:00:00+00:00,16721.27,16772.01,16704.07,16735.11,23267.94119,NaN,NaN,False,NaN,NaN,False,False,16721.28,67.94,50.73,17.21,67.94,NaN,NaN
9,2023-01-02 12:00:00+00:00,16735.51,16750.00,16669.15,16734.66,21747.21491,NaN,NaN,False,NaN,NaN,False,False,16735.11,80.85,14.89,65.96,80.85,NaN,NaN


In [29]:
in_position = False
entry_price = None
stop_loss = None
take_profit = None
trade_log = []


for i, row in df.iterrows():
    if in_position == False:
        if row['Real_Breakout'] == True and pd.notna(row['ATR_Shifted']):
            entry_price = row['Close']
            entry_time = row['Timestamp']
            stop_loss = entry_price - (row['ATR_Shifted'] * 1)
            take_profit = entry_price + (row['ATR_Shifted'] * 2)
            in_position = True
    else:
        if row['Low'] <= stop_loss:
            trade_log.append({
                'entry_time': entry_time,
                'entry_price': entry_price,
                'exit_time': row['Timestamp'],
                'exit_price': stop_loss,
                'exit_reason': 'stop_loss',
                'pnl': stop_loss - entry_price,
                'pnl_pct': (stop_loss - entry_price) / entry_price
            })
            in_position = False
        elif row['High'] >= take_profit:
            trade_log.append({
                'entry_time': entry_time,
                'entry_price': entry_price,
                'exit_time': row['Timestamp'],
                'exit_price': take_profit,
                'exit_reason': 'take_profit',
                'pnl': take_profit - entry_price,
                'pnl_pct': (take_profit - entry_price) / entry_price
            })
            in_position = False

In [30]:
pd.DataFrame(trade_log)

,entry_time,entry_price,exit_time,exit_price,exit_reason,pnl,pnl_pct
0,2023-01-04 00:00:00+00:00,16862.02,2023-01-04 12:00:00+00:00,16772.621429,stop_loss,-89.398571,-0.005302
1,2023-01-09 00:00:00+00:00,17197.00,2023-01-09 16:00:00+00:00,17355.078571,take_profit,158.078571,0.009192
2,2023-01-10 12:00:00+00:00,17327.70,2023-01-11 16:00:00+00:00,17548.541429,take_profit,220.841429,0.012745
3,2023-01-11 20:00:00+00:00,17943.26,2023-01-12 00:00:00+00:00,18197.532857,take_profit,254.272857,0.014171
4,2023-01-12 16:00:00+00:00,18885.35,2023-01-13 12:00:00+00:00,19308.434286,take_profit,423.084286,0.022403
...,...,...,...,...,...,...,...
271,2026-06-08 12:00:00+00:00,63774.48,2026-06-09 00:00:00+00:00,62640.137857,stop_loss,-1134.342143,-0.017787
272,2026-06-14 20:00:00+00:00,65746.45,2026-06-15 12:00:00+00:00,66739.248571,take_profit,992.798571,0.015100
273,2026-06-22 12:00:00+00:00,64836.95,2026-06-22 20:00:00+00:00,64204.801429,stop_loss,-632.148571,-0.009750
274,2026-07-02 12:00:00+00:00,61612.93,2026-07-05 20:00:00+00:00,63595.537143,take_profit,1982.607143,0.032178


In [31]:
trades_df = pd.DataFrame(trade_log)

In [32]:
trades_df['exit_reason'].value_counts()

exit_reason
stop_loss      169
take_profit    107
Name: count, dtype: int64

In [33]:
trades_df['pnl_pct'].sum() * 100

np.float64(45.50225281080444)

In [34]:
trades_df['cumulative_equity'] = (1 + trades_df['pnl_pct']).cumprod()
trades_df['cumulative_equity']

0      0.994698
1      1.003842
2      1.016636
3      1.031042
4      1.054141
         ...   
271    1.469752
272    1.491946
273    1.477400
274    1.524940
275    1.511255
Name: cumulative_equity, Length: 276, dtype: float64

In [35]:
trades_df['running_peak'] = trades_df['cumulative_equity'].cummax()
trades_df['running_peak']

0      0.994698
1      1.003842
2      1.016636
3      1.031042
4      1.054141
         ...   
271    1.895796
272    1.895796
273    1.895796
274    1.895796
275    1.895796
Name: running_peak, Length: 276, dtype: float64

In [36]:
trades_df['drawdown_pct'] = (trades_df['cumulative_equity'] - trades_df['running_peak']) / trades_df['running_peak']
trades_df['drawdown_pct']

0      0.000000
1      0.000000
2      0.000000
3      0.000000
4      0.000000
         ...   
271   -0.224731
272   -0.213024
273   -0.220697
274   -0.195620
275   -0.202839
Name: drawdown_pct, Length: 276, dtype: float64

In [37]:
max_drawdown = trades_df['drawdown_pct'].min()
max_drawdown

-0.22473093102806171

In [38]:
mean_return = trades_df['pnl_pct'].mean()
std_return = trades_df['pnl_pct'].std()

trades_per_year = len(trades_df) / 3.5

sharpe_per_trade = mean_return / std_return
sharpe_annualized = sharpe_per_trade * np.sqrt(trades_per_year)
sharpe_annualized

np.float64(0.8365711051978623)

In [39]:
mean_return = trades_df['pnl_pct'].mean()
downside_std = trades_df[trades_df['pnl_pct'] < 0]['pnl_pct'].std()

sortino_per_trade = mean_return / downside_std
sortino_annualized = sortino_per_trade * np.sqrt(trades_per_year)
sortino_annualized

np.float64(3.0409200741368414)

In [40]:
trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276 entries, 0 to 275
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   entry_time         276 non-null    datetime64[ns, UTC]
 1   entry_price        276 non-null    float64            
 2   exit_time          276 non-null    datetime64[ns, UTC]
 3   exit_price         276 non-null    float64            
 4   exit_reason        276 non-null    object             
 5   pnl                276 non-null    float64            
 6   pnl_pct            276 non-null    float64            
 7   cumulative_equity  276 non-null    float64            
 8   running_peak       276 non-null    float64            
 9   drawdown_pct       276 non-null    float64            
dtypes: datetime64[ns, UTC](2), float64(7), object(1)
memory usage: 21.7+ KB


In [41]:
trades_df['weekday'] = trades_df['entry_time'].dt.day_name()
trades_df['hour'] = trades_df['entry_time'].dt.hour

In [42]:
trades_df.groupby('weekday')['pnl_pct'].mean().sort_values()

weekday
Saturday    -0.016185
Tuesday     -0.000902
Monday       0.000487
Friday       0.000746
Wednesday    0.002936
Sunday       0.006415
Thursday     0.006637
Name: pnl_pct, dtype: float64

In [43]:
trades_df.groupby('hour')['pnl_pct'].mean().sort_values()

hour
16    0.000007
12    0.000234
8     0.001344
0     0.001680
4     0.002560
20    0.007667
Name: pnl_pct, dtype: float64

In [44]:
wins = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct']
losses = trades_df[trades_df['pnl_pct'] < 0]['pnl_pct']

win_rate = len(wins) / len(trades_df)
profit_factor = wins.sum() / abs(losses.sum())

max_drawdown = trades_df['drawdown_pct'].min()

summary_kpis = {
    'total_trades': len(trades_df),
    'win_rate_pct': round(win_rate * 100, 2),
    'profit_factor': round(profit_factor, 2),
    'max_drawdown_pct': round(max_drawdown * 100, 2),
    'sharpe_ratio': round(sharpe_annualized, 2),
    'sortino_ratio': round(sortino_annualized, 2),
    'total_return_pct': round(trades_df['pnl_pct'].sum() * 100, 2),
    'compounded_return_pct': round((trades_df['cumulative_equity'].iloc[-1] - 1) * 100, 2),
    'avg_win_pct': round(wins.mean() * 100, 2),
    'avg_loss_pct': round(losses.mean() * 100, 2),
}

summary_df = pd.DataFrame([summary_kpis])
summary_df

,total_trades,win_rate_pct,profit_factor,max_drawdown_pct,sharpe_ratio,sortino_ratio,total_return_pct,compounded_return_pct,avg_win_pct,avg_loss_pct
0,276,38.77,1.24,-22.47,0.84,3.04,45.5,51.13,2.22,-1.14


In [45]:
summary_df.to_csv('..\\data\\processed\\summary-kpis.csv', index=False)

In [46]:
trades_df.to_csv('..\\data\\processed\\backtested.csv', index=False)